# Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 29.04.2024

## 1. Setup

### 1.1 Import Libraries

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scgpt as scg
import squidpy as sq
import torch
from datasets import load_from_disk
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils

### 1.2 Run Notebook Setup

In [13]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### 1.3 Configure Paths and Create Directories

In [14]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
cell_gene_means_dictionary_file = "../datasets/cell_gene_means_dictionary.pkl"
cell_gene_reg_stds_dictionary_file = "../datasets/cell_gene_reg_stds_dictionary.pkl"
neighborhood_gene_means_dictionary_file = "../datasets/neighborhood_gene_means_dictionary.pkl"
neighborhood_gene_reg_stds_dictionary_file = "../datasets/neighborhood_gene_reg_stds_dictionary.pkl"
geneformer_token_dictionary_file = "../datasets/geneformer_token_dictionary.pkl"
nichejepa_token_dictionary_file = "../datasets/token_dictionary.pkl"

## 2. Tokenization

In [15]:
dataset_name = "nanostring_cosmx_human_brain"

### 2.1 Create Token Dictionary

In [6]:
# Create token dict for nichejepa with cell and neighborhood tokens
with open(geneformer_token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

new_token_dict = {
    (key + "_cell" if "ENS" in key else key): value for key, value in token_dict.items()}
new_token_dict_copy = new_token_dict.copy()
i = 0
for key in new_token_dict_copy.keys():
    if "ENS" in key:
        new_token_dict[key.replace("cell", "neighborhood")] = 25426 + i
        i += 1
del(new_token_dict_copy)

with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

In [7]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

### 2.2 Compute Normalization Factors for Gene Counts

#### 2.2.1 Compute Corpus Means and Regularized Standard Deviations for Cell and Neighborhood Genes

In [8]:
# Load data
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}.h5ad")

In [9]:
# Normalize cell gene counts by read depth
read_depth_norm_counts = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000

# Compute means and vars of read-depth normalized cell gene counts
cell_gene_means = np.mean(read_depth_norm_counts, axis=0)
cell_gene_vars = np.var(read_depth_norm_counts, axis=0)

In [10]:
# Normalize neighborhood gene counts by read depth
read_depth_norm_counts_neighborhood = (adata.layers["counts_neighborhood"] /
                                       adata.layers["counts_neighborhood"].sum(1).reshape(-1, 1)
                                       * 10000)

# Compute means and vars of read-depth normalized neighborhood gene counts
neighborhood_gene_means = np.mean(read_depth_norm_counts_neighborhood, axis=0)
neighborhood_gene_vars = np.var(read_depth_norm_counts_neighborhood, axis=0)

In [11]:
# Learn mean-variance relationships for cell and neighborhood genes respectively
# to remove (only) expected variance based on mean to stabilize variances across genes
not_const_cell_genes = cell_gene_vars > 0
cell_gene_est_log_vars = np.zeros(adata.shape[1], dtype=np.float64)
y = np.log10(cell_gene_vars[not_const_cell_genes])
x = np.log10(cell_gene_means[not_const_cell_genes])
cell_gene_model = loess(x, y, span=0.3, degree=2)
cell_gene_model.fit()
cell_gene_est_log_vars[not_const_cell_genes] = cell_gene_model.outputs.fitted_values
cell_gene_reg_stds = np.sqrt(10**cell_gene_est_log_vars)

not_const_neighborhood_genes = neighborhood_gene_vars > 0
neighborhood_gene_est_log_vars = np.zeros(adata.shape[1], dtype=np.float64)
y = np.log10(neighborhood_gene_vars[not_const_neighborhood_genes])
x = np.log10(neighborhood_gene_means[not_const_neighborhood_genes])
neighborhood_gene_model = loess(x, y, span=0.3, degree=2)
neighborhood_gene_model.fit()
neighborhood_gene_est_log_vars[not_const_neighborhood_genes] = neighborhood_gene_model.outputs.fitted_values
neighborhood_gene_reg_stds = np.sqrt(10**neighborhood_gene_est_log_vars)

In [12]:
# Create norm factor dictionaries for tokenizer and store on disk
cell_gene_means_dict = {}
cell_gene_reg_stds_dict = {}
neighborhood_gene_means_dict = {}
neighborhood_gene_reg_stds_dict = {}

for gene, cell_gene_mean, cell_gene_reg_std, neighborhood_gene_mean, neighborhood_gene_reg_std in zip(
        adata.var["ensembl_id"].values,
        cell_gene_means,
        cell_gene_reg_stds,
        neighborhood_gene_means,
        neighborhood_gene_reg_stds):
    cell_gene_means_dict[gene] = cell_gene_mean
    cell_gene_reg_stds_dict[gene] = cell_gene_reg_std
    neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
    neighborhood_gene_reg_stds_dict[gene] = neighborhood_gene_reg_std

with open(cell_gene_means_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_means_dict, file)
with open(cell_gene_reg_stds_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_reg_stds_dict, file)
with open(neighborhood_gene_means_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_means_dict, file)
with open(neighborhood_gene_reg_stds_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_reg_stds_dict, file)

#### 2.2.2 Test Normalization

In [14]:
# Keep only protein-coding and miRNA genes
vocab = list(token_dict.keys())
coding_miRNA_ids = [
    key.split("_")[0] for key in list(vocab) if "_cell" in key]
coding_miRNA_dict = dict(zip(coding_miRNA_ids, [True] * len(vocab)))
coding_miRNA_idx = np.where(
    [coding_miRNA_dict.get(gene_id, False) for gene_id in adata.var["ensembl_id"]])[0]
coding_miRNA_adata = adata[:, coding_miRNA_idx]

In [15]:
# Load cell and neighborhood gene corpus means and vars
with open(cell_gene_means_dictionary_file, "rb") as f:
    cell_gene_means_dict = pickle.load(f)
with open(cell_gene_reg_stds_dictionary_file, "rb") as f:
    cell_gene_reg_stds_dict = pickle.load(f)
with open(neighborhood_gene_means_dictionary_file, "rb") as f:
    neighborhood_gene_means_dict = pickle.load(f)
with open(neighborhood_gene_reg_stds_dictionary_file, "rb") as f:
    neighborhood_gene_reg_stds_dict = pickle.load(f)

cell_gene_means = np.array(
    [cell_gene_means_dict[gene_id] for gene_id in adata.var["ensembl_id"][coding_miRNA_idx]])
cell_gene_reg_stds = np.array(
    [cell_gene_reg_stds_dict[gene_id] for gene_id in adata.var["ensembl_id"][coding_miRNA_idx]])
neighborhood_gene_means = np.array(
    [neighborhood_gene_means_dict[gene_id] for gene_id in adata.var["ensembl_id"][coding_miRNA_idx]])
neighborhood_gene_reg_stds = np.array(
    [neighborhood_gene_reg_stds_dict[gene_id] for gene_id in adata.var["ensembl_id"][coding_miRNA_idx]])

In [17]:
# Compute mean std normalized counts where std is exc
read_depth_norm_counts = coding_miRNA_adata.X / coding_miRNA_adata.X.sum(1).reshape(-1, 1) * 10000
mean_std_norm_counts = (read_depth_norm_counts - cell_gene_means) / cell_gene_reg_stds

In [18]:
read_depth_norm_counts_neighborhood = (
    coding_miRNA_adata.layers["counts_neighborhood"] /
    coding_miRNA_adata.layers["counts_neighborhood"].sum(1).reshape(-1, 1) * 10000)
mean_std_norm_counts_neighborhood = (
    read_depth_norm_counts_neighborhood - neighborhood_gene_means) / neighborhood_gene_reg_stds

### 2.3 Run Tokenizer

In [9]:
tk = tokenizers.STRankTokenizer(
    cell_gene_means_file=cell_gene_means_dictionary_file,
    cell_gene_reg_stds_file=cell_gene_reg_stds_dictionary_file,
    neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file,
    neighborhood_gene_reg_stds_file=neighborhood_gene_reg_stds_dictionary_file,
    token_dictionary_file=nichejepa_token_dictionary_file,
    custom_attr_name_dict={"cell_type": "cell_types",
                           "niche_type": "niche_types"})

In [ ]:
tk.tokenize_data(input_directory=st_data_silver_folder_path,
                 output_directory=st_data_gold_folder_path,
                 output_file_prefix=dataset_name)

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain.h5ad'...
'../datasets/st_data/silver/nanostring_cosmx_human_brain.h5ad' has no column 'filter_pass'; tokenizing all cells.
Creating Hugging Face dataset...


Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Saving the dataset (0/7 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

### 2.4 Evaluate Tokenizer

In [16]:
# Load tokenized dataset
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{dataset_name}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset["input_ids"][0]

In [ ]:
# Get cell and neighborhood gene token rankings
gene_token_ranking = np.array(dataset["input_ids"])
adata.obsm["cell_gene_token_ranking"] = gene_token_ranking[:, :1024]
adata.obsm["neighborhood_gene_token_ranking"] = gene_token_ranking[:, 1024:]

#### 2.5.1 Neighborhood Ranking

In [ ]:
# Normalize counts as per https://www.sc-best-practices.org/preprocessing_visualization/normalization.html
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)

In [ ]:
# Compute spatial autocorrelation score
sq.gr.spatial_autocorr(adata)
if "I" in adata.var.columns:
    del(adata.var["I"])
adata.var = adata.var.join(adata.uns["moranI"][["I"]], how="left")

In [ ]:
adata.obsm["neighborhood_gene_token_ranking"]

In [ ]:
adata.obsm["neighborhood_gene_token_ranking"]

In [ ]:
adata.obsm["spatial"]

In [ ]:
adata.uns["moranI"]

In [ ]:
adata2 = sq.datasets.visium_hne_adata()
adata2

In [ ]:
adata2.X.toarray()

In [ ]:
sq.pl.spatial_scatter(adata, layer="counts", color="MBP", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, layer="counts_neighborhood", color="MBP", shape=None) 

In [ ]:
np.log(700)

In [ ]:
sq.pl.spatial_scatter(adata, color="MBP", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, layer="counts", color="MBP", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, layer="counts_neighborhood", color="MBP", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, color="NGF", shape=None) 

In [ ]:
norm_counts.shape

In [ ]:
len(dataset["input_ids"])

In [ ]:
adata.layers

In [ ]:
adata.layers["counts_neighborhood"]

In [ ]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

In [ ]:
token_dict_reverse = {value: key for key, value in token_dict.items()}

In [ ]:
# Sum up ranks for each neighborhood gene token
neighborhood_gene_token_rank_sums = {}

for i, row in enumerate(adata.obsm["neighborhood_gene_token_ranking"]):
    for rank, token in enumerate(row[:10]):
        if token in neighborhood_gene_token_rank_sums:
            neighborhood_gene_token_rank_sums[token] += (10 - rank)
        else:
            neighborhood_gene_token_rank_sums[token] = (10 - rank)

# Replace gene token by gene ensembl id
neighborhood_gene_rank_sums = {}

for key, value in token_dict.items():
    if value in neighborhood_gene_token_rank_sums:
        neighborhood_gene_rank_sums[key.replace("_neighborhood", "")] = neighborhood_gene_token_rank_sums[value]

neighborhood_gene_rank_file = f"../datasets/temp/{dataset_name}/neighborhood_gene_rank_sums.pkl"

with open(neighborhood_gene_rank_file, "wb") as file:
    pickle.dump(neighborhood_gene_rank_sums, file)

In [ ]:
# Sum up ranks for each cell gene token
cell_gene_token_rank_sums = {}

for i, row in enumerate(adata.obsm["cell_gene_token_ranking"]):
    for rank, token in enumerate(row[:10]):
        if token in cell_gene_token_rank_sums:
            cell_gene_token_rank_sums[token] += (10 - rank)
        else:
            cell_gene_token_rank_sums[token] = (10 - rank)

# Replace gene token by gene ensembl id
cell_gene_rank_sums = {}

for key, value in token_dict.items():
    if value in cell_gene_token_rank_sums:
        cell_gene_rank_sums[key.replace("_cell", "")] = cell_gene_token_rank_sums[value]

cell_gene_rank_file = f"../datasets/temp/{dataset_name}/cell_gene_rank_sums.pkl"

with open(cell_gene_rank_file, "wb") as file:
    pickle.dump(cell_gene_rank_sums, file)

In [ ]:
adata.var["neigborhood_gene_rank_sums"] = adata.var["ensembl_id"].map(neighborhood_gene_rank_sums)
adata.var["cell_gene_rank_sums"] = adata.var["ensembl_id"].map(cell_gene_rank_sums)

In [ ]:
adata.var["neigborhood_gene_rank_sums"] = adata.var["neigborhood_gene_rank_sums"].fillna(0)
adata.var["cell_gene_rank_sums"] = adata.var["cell_gene_rank_sums"].fillna(0)

In [ ]:
correlation, p_value = spearmanr(adata.var["neigborhood_gene_rank_sums"],
                                 adata.var["I"])
print("Spearman Correlation Coefficient:", correlation)
print("P-value:", p_value)

In [ ]:
adata.var["neigborhood_gene_rank_sums"]["MBP"]

In [ ]:
np.max(adata.var["neigborhood_gene_rank_sums"])

In [ ]:
adata.var["neigborhood_gene_rank_sums"].sort_values(ascending=False)

In [ ]:
adata.uns["moranI"]

In [ ]:
running_corr = []
running_p_value = []

for i in range(norm_counts_neighborhood.shape[0]):
    i_corr, i_p_value = spearmanr(norm_counts_neighborhood[i],
                                  adata.var["I"])
running_corr.append(i_corr)
running_p_value.append(i_p_value)

corr = np.mean(running_corr)

print("Spearman Correlation Coefficient:", corr)
#print("P-value:", p_value)

In [ ]:
"counts_neighborhood" in adata.layers.keys()

In [ ]:
norm_counts_neighborhood

In [ ]:
correlation, p_value = spearmanr(adata.var["cell_gene_rank_sums"],
                                 adata.var["I"])
print("Spearman Correlation Coefficient:", correlation)
print("P-value:", p_value)

In [ ]:
correlation, p_value = spearmanr(adata.var["cell_gene_rank_sums"],
                                 adata.var["neigborhood_gene_rank_sums"])
print("Pearson Correlation Coefficient:", correlation)
print("P-value:", p_value)

In [ ]:
adata.var

In [ ]:
adata.uns["moranI"]

In [ ]:
adata.X = norm_counts

In [ ]:
len(dataset["gene_tokens_cell"][0])

In [ ]:
len(dataset["gene_tokens_neighborhood"][0])

In [ ]:
type(dataset["gene_tokens_neighborhood"])

In [ ]:
emsemblid_embedding_mapping["ENSG00000288721"].shape

In [ ]:
# Get input tokens from token ids
def get_tokens(examples):
    examples["input_tokens"] = []
    token_sequences = examples["input_ids"]
    for token_seq in token_sequences:
        tokens = []
        for token_id in token_seq:
            ensembl_id = token_dict_reverse[token_id].replace("_cell", "").replace("_neighborhood", "")
            tokens.append(emsemblid_embedding_mapping.get(ensembl_id, np.zeros(512, dtype=np.float32)))
        examples["input_tokens"].append(tokens)
    return examples

encoded_dataset = dataset.map(get_tokens, batched=True, batch_size=2)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(dataset["input_ids"], dataset["cell_type"], test_size=0.1, random_state=42, stratify=dataset["cell_type"])

In [ ]:
encoded_dataset["input_tokens"][0][:20]

In [ ]:
2+3

In [ ]:
new_folder_path = tokenized_data_folder_path.replace("nanostring_cosmx_human_brain", "nanostring_cosmx_human_brain_processed")

In [ ]:
encoded_dataset.save_to_disk(new_folder_path)

In [ ]:
encoded_dataset["input_tokens"][0]

In [ ]:
# Get scGPT gene embeddings for each gene Ensembl id
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
n_bins = 51
pad_value = -2
n_input_bins = n_bins

model_dir = Path("../scgpt/pretrained_model")
model_config_file = model_dir / "args.json"
model_file = model_dir / "best_model.pt"
vocab_file = model_dir / "vocab.json"

vocab = GeneVocab.from_file(vocab_file)
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# Retrieve model parameters from config files
with open(model_config_file, "r") as f:
    model_configs = json.load(f)
print(
    f"Resume model from {model_file}, the model args will override the "
    f"config {model_config_file}."
)
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
n_layers_cls = model_configs["n_layers_cls"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    vocab=vocab,
    pad_value=pad_value,
    n_input_bins=n_input_bins,
)

try:
    model.load_state_dict(torch.load(model_file))
    print(f"Loading all model params from {model_file}")
except:
    # only load params that are in the model and match the size
    model_dict = model.state_dict()
    pretrained_dict = torch.load(model_file)
    pretrained_dict = {
        k: v
        for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)

model.to(device)

gene2idx = vocab.get_stoi()

# Create mapping from Ensembl id to scGPT token id
metadata_df = pd.read_csv(Path("../scgpt/metadata/gene_info.csv"))

ensemblid_mapping = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))
ensemblid2idx = {ensemblid_mapping[k]: v for k, v in gene2idx.items() if k in ensemblid_mapping}

# Retrieve the data-independent gene embeddings from scGPT
gene_ids = np.array([id for id in ensemblid2idx.values()])
gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
gene_embeddings = gene_embeddings.detach().cpu().numpy()

emsemblid_embedding_mapping = {}
for i, (key, val) in enumerate(ensemblid2idx.items()):
    emsemblid_embedding_mapping[key] = gene_embeddings[i, :]

In [ ]:
len(ensemblid2idx)

In [ ]:
gene_embeddings.shape

In [ ]:
emsembl_embedding_mapping

In [ ]:
gene_ids

In [ ]:
gene_embeddings

In [ ]:
# Filter on the intersection between the Immune Human HVGs found in step 1.2 and scGPT's 30+K foundation model vocab
gene_embeddings = {gene: gene_embeddings[i] for i, gene in enumerate(ensemblid2idx.keys()) if gene in adata.var.index.tolist()}
print('Retrieved gene embeddings for {} genes.'.format(len(gene_embeddings)))

In [ ]:
train_dataset = encoded_dataset['train']
test_dataset = encoded_dataset['test']

scores = dict()
for i in tqdm(range(model.config.n_layer + 1)):
    # fit linear classifier
    lr_clf = LogisticRegression(max_iter=1000)
    lr_clf.fit(train_dataset[f'features_{i}'], train_dataset['label'])
    # compute accuracy on training + test set
    training_score = lr_clf.score(train_dataset[f'features_{i}'], train_dataset['label'])
    test_score = lr_clf.score(test_dataset[f'features_{i}'], test_dataset['label'])
    scores[f'features_{i}'] = (training_score, test_score)

#### 2.5.1 Neighborhood Ranking